# Failure analysis and safe recovery

**Time:** 90–150 minutes  
**Prerequisites:** notebook 04 and at least one successful or failed run.

## Learning objectives

- Locate the first failed pipeline stage from saved evidence
- Distinguish perception, planning, reachability, pickup, placement, and support failures
- Choose safe retry, recapture, replan, or stop behavior
- Preserve confidence and physical gates instead of weakening them

> A recovery policy must make the system safer and more observable. ‘Try again forever’ and ‘lower the threshold’ are not recovery strategies.

In [ ]:
import json
from collections import Counter
from pathlib import Path

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / 'fanuc_er4ia_sim' / 'sort_scene.py').is_file(): return p
    raise FileNotFoundError('Repository root not found')
ROOT = find_root(); SIM = ROOT / 'fanuc_er4ia_sim'; RUNS = SIM / 'runs'
runs = sorted((p for p in RUNS.glob('*') if p.is_dir()), key=lambda p: p.stat().st_mtime, reverse=True) if RUNS.exists() else []
RUN = runs[0] if runs else RUNS / 'stack_01'
print('Selected run:', RUN)

## 1. Find the first missing or failed stage

Use this order: capture → Cosmos → plan → execute → verify. Later symptoms are often consequences. For example, ‘zero placements’ is not yet a root cause if the planner produced zero eligible picks.

In [ ]:
def load(run, name):
    path = run / name
    try: return json.loads(path.read_text())
    except (FileNotFoundError, json.JSONDecodeError): return None

def diagnose(run):
    if not (run / 'world_camera.png').exists(): return 'capture', 'Image missing: inspect Isaac/GPU/camera output.'
    det = load(run, 'detections.json')
    if det is None: return 'perception', 'Parsed detections missing: inspect detections_raw.txt and Cosmos logs.'
    plan = load(run, 'sort_plan.json')
    if plan is None: return 'planning', 'Plan missing: rerun planner and read its exception.'
    if not plan.get('picks'): return 'planning gates', f"No eligible picks; skipped={Counter(o.get('skip_reason','unknown') for o in plan.get('skipped', []))}"
    report = load(run, 'execution_report.json')
    if report is None: return 'execution', 'Report missing even if Isaac exited zero: inspect simulator output.'
    if report.get('task_complete'): return 'complete', 'All report gates passed.'
    return 'verification', report.get('abort_reason') or 'Inspect the first unverified pickup/placement.'

print(diagnose(RUN))

## 2. Read planner rejections as explanations

Each skipped observation should contain a reason. Grouping those reasons often reveals one shared cause: low confidence, incomplete colour inventory, duplicate detections, poor image fit, or unsafe geometry.

In [ ]:
plan = load(RUN, 'sort_plan.json') or {}
reasons = Counter(o.get('skip_reason', 'unknown') for o in plan.get('skipped', []))
print('Eligible picks:', len(plan.get('picks', [])))
print('Skip reasons:', reasons)
for obj in plan.get('skipped', []):
    print(obj.get('detection_id'), obj.get('category'), obj.get('colour'), '->', obj.get('skip_reason'))

## 3. Find the first physical failure

For a tower, later layers depend on earlier support. Diagnose the first failed event, not the most dramatic final pose.

In [ ]:
report = load(RUN, 'execution_report.json') or {}
for i, event in enumerate(report.get('picks', [])):
    status = {k: event.get(k) for k in ['pickup_verified','placement_verified','placement_pose_verified','stack_support_verified','placement_error_mm','placement_yaw_error_deg_mod90','placement_tilt_deg']}
    print('event', i, status)
    if not event.get('placement_verified') or event.get('stack_support_verified') is False:
        print('FIRST FAILURE FULL RECORD:\n', json.dumps(event, indent=2))
        break

## 4. Match the recovery to the evidence

- **Capture missing/corrupt:** stop, check GPU/camera, capture into a new directory.
- **Malformed Cosmos JSON:** preserve raw reply; retry inference a bounded number of times or improve the schema prompt. Never invent missing fields.
- **Low confidence or incomplete inventory:** recapture if occluded; otherwise stop for review. Do not lower gates.
- **Unreachable/collision-risk plan:** reject it and revise scene/planner offline. Never command the nearest reachable approximation.
- **Pickup failure:** stop the tower, inspect alignment/contact and the video; a blind retry may move the object.
- **Placement/support failure:** stop immediately. Later placements assume a valid preceding layer.
- **Missing report:** treat as failure even when Isaac returns code 0.

In [ ]:
RECOVERY_POLICY = {
    'capture': {'max_retries': 1, 'action': 'recapture_new_run', 'requires_review': True},
    'perception': {'max_retries': 2, 'action': 'retry_same_frozen_image', 'requires_review': True},
    'planning gates': {'max_retries': 0, 'action': 'stop_and_diagnose', 'requires_review': True},
    'execution': {'max_retries': 0, 'action': 'stop_and_inspect_logs', 'requires_review': True},
    'verification': {'max_retries': 0, 'action': 'stop_and_inspect_first_event', 'requires_review': True},
}
stage, detail = diagnose(RUN)
print('Stage:', stage); print('Evidence:', detail); print('Policy:', RECOVERY_POLICY.get(stage, {'action': 'none'}))

## 5. Failure drill

In pairs, choose three existing runs: one successful, one perception/planning failure, and one execution failure. The person diagnosing may inspect only artifacts and logs at first; the partner knows what was changed. Record: first failed stage, evidence, ruled-out causes, safe next action, and evidence required before resuming.

If you lack a failed execution run, use a deliberately difficult **simulation-only** randomized scene from notebook 07. Do not tamper with an executable plan to manufacture a dangerous motion.

## Checkpoint

You are finished when your team can produce a one-sentence diagnosis in this form:

> The first failure occurred at **[stage]** because **[specific artifact evidence]**; the safe next action is **[bounded action]**, and we will resume only after **[verification]**.

A simulator process exit code is never sufficient evidence of success.